# Fig6, stage 04: clean, transform, embed and cluster

5 things happen here, in this order.

1. Clean.  Drop a cell that sits above the 99.9997th percentile of any
   channel, which removes the extreme upper tail, then drop a cell with no
   signal in any channel.
2. Build a second view of the same cells, called `positives`, in which each
   channel has had its background cutoff subtracted, negative values set to
   zero, and the rest scaled to run from zero to one.  A value above zero in
   this view means the marker is really there, which the logicle values cannot
   say on their own, because the logicle scale has no fixed zero point.
3. Transform.  Apply the logicle transformation channel by channel.  Logicle is
   linear near zero and logarithmic further out, so it keeps the negative and
   near-zero values that a plain logarithm cannot show.
4. Embed.  Principal components, a neighbour graph and a UMAP, on the
   downsampled subset.
5. Cluster.  Leiden on that graph, at resolution 0.9.

## Inputs

| file | produced by | holds |
|---|---|---|
| `Fig6_Gated.h5ad` | stage 02 | the gated cells |
| `marker_cutoff.pkl` | stage 03 of Loop0 | one background cutoff per antibody |
| `logicle_parameters.json` | stage 03 of Loop0 | the logicle parameters `t`, `m`, `w`, `a` |

## Outputs

| file | holds |
|---|---|
| `Fig6_Raw_Pos.h5ad` | the cleaned cells, with a `raw` layer and a `positives` layer, before the logicle transformation |
| `Fig6_logicle.h5ad` | the same cells with logicle-transformed intensities in `.X` |
| `Fig6_logicle_500k.h5ad` | about 500,000 of those cells |
| `Fig6_UMAP.h5ad` | the same subset with principal components, a neighbour graph and a UMAP |
| `Fig6_Leiden.h5ad` | the same subset with Leiden clusters |

Every step records what it did in `adata.uns["processing_steps"]`, so a written
file says how it was made.

Run the cells from top to bottom.  The logicle transformation is the long step,
at roughly ten minutes per channel for 20 channels.

In [ ]:
import gc
import json
import pickle
import sys
import time

import anndata as ann
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.colors import LinearSegmentedColormap

sys.path.append("..")  # labcompass.py sits beside the loop folders
from labcompass import (
    clean_intensities,
    compute_umap,
    downsample,
    logicle_transform,
    plot_marker,
    plot_umap_by_group,
    plot_leiden_clusters,
    rotate_umap,
)

# ------------------------------------------------------------------- input
IN_GATED = "Fig6_Gated.h5ad"
CUTOFF_FILE = "marker_cutoff.pkl"
LOGICLE_FILE = "logicle_parameters.json"

# ------------------------------------------------------------------ output
OUT_RAW_POS = "Fig6_Raw_Pos.h5ad"
OUT_LOGICLE = "Fig6_logicle.h5ad"
OUT_LOGICLE_SUBSET = "Fig6_logicle_500k.h5ad"
OUT_UMAP = "Fig6_UMAP.h5ad"
OUT_LEIDEN = "Fig6_Leiden.h5ad"

# -------------------------------------------------------------- parameters
# A cell above this percentile in any channel is dropped as an extreme value.
PERCENTILE = 99.9997
DESIRED_CELL_NUMBER = 500000
DOWNSAMPLE_SEED = 0

N_PCS = 19
N_NEIGHBORS = 15
# scanpy's default random state.  The published embeddings were made with it,
# so leaving it at 0 is what reproduces them.
EMBEDDING_RANDOM_STATE = 0
# Rotation only turns the picture.  It changes no distance and no cluster.
UMAP_ROTATION = 45
LEIDEN_RESOLUTION = 0.9
# Seed for Leiden.  The published clusters were made with this value.
LEIDEN_SEED = 51

# Marker order used in the figure panels.
MARKERS = [
    "CD34", "CD45RA", "CD90", "EPCR", "CD235a", "CD41a",
    "CD117", "HLADR", "CD14", "CD16", "CD10", "CD7",
    "CD19", "CD49c", "CD38", "CD135", "CD56", "CD15",
    "CD123", "CD45",
]
# In the zero-to-one "positives" view, anything above this counts as expressed.
POSITIVE_THRESHOLD = 0.01

## Read the gated cells and the calibration

In [ ]:
adata = sc.read(IN_GATED)
adata.uns["processing_steps"] = {}
print(f"{adata.n_obs:,} cells x {adata.n_vars} antibodies")
adata

The calibration was made once, in Loop0, from the fluorescence-minus-one
controls.  Every loop uses the same copy, so the intensity scale is comparable
across loops.  The check below fails loudly if the channel order of the data and
of the calibration have drifted apart, because a silent mismatch would apply
each antibody's cutoff to the wrong channel.

In [ ]:
with open(LOGICLE_FILE) as handle:
    logicle = json.load(handle)
t, m, w, a = logicle["t"], logicle["m"], logicle["w"], logicle["a"]

with open(CUTOFF_FILE, "rb") as handle:
    marker_cutoff = pickle.load(handle)

if list(adata.var_names) != list(marker_cutoff):
    raise ValueError(
        "the channel order of the data and of marker_cutoff.pkl differ:\n"
        f"  data:   {list(adata.var_names)}\n"
        f"  cutoff: {list(marker_cutoff)}"
    )
if len(t) != adata.n_vars:
    raise ValueError(f"{len(t)} logicle parameters for {adata.n_vars} channels")

marker_cutoff

## Step 1, clean

Two filters, in this order.  First drop a cell if any channel is above the
99.9997th percentile of that channel.  Then drop a cell if no channel is above
zero.

In [ ]:
before = adata.n_obs
clean_intensities(adata, marker_cutoff, percentile=True, p=PERCENTILE, zero_qc=True)
adata.uns["processing_steps"]["marker_cutoff"] = "99.5 per cent of the FMO"
adata.uns["processing_steps"]["percentile"] = f"{PERCENTILE} per cent"
adata.uns["processing_steps"]["zero_QC"] = "cells with no positive channel removed"

adata.X = np.nan_to_num(adata.X, nan=0.0)
print(f"{before:,} cells -> {adata.n_obs:,} cells")
adata.uns["processing_steps"]

## Step 2, the `positives` view

Stored as a layer, so its cells line up one to one with `.X`.  The marker panels
further down read it through `adata.raw`.

In [ ]:
positives = adata.copy()
clean_intensities(
    positives,
    marker_cutoff,
    subtract_fmo_cutoff=True,
    negatives_to_zeroes=True,
    scaling=True,
)

adata.layers["raw"] = adata.X.copy()
adata.layers["positives"] = positives.X.copy()
adata.uns["processing_steps"]["positives_layer"] = (
    "FMO cutoff subtracted, negatives zeroed, each channel scaled to 0-1"
)

del positives
gc.collect()
print("positives layer runs from", adata.layers["positives"].min(),
      "to", adata.layers["positives"].max())

In [ ]:
adata.write(OUT_RAW_POS)
print(f"wrote {OUT_RAW_POS}: {adata.n_obs:,} cells")

## Step 3, the logicle transformation

One channel at a time, so only one column is held in memory at once.

In [ ]:
start = time.perf_counter()
logicle_transform(adata, t, m, w, a)
elapsed = time.perf_counter() - start

adata.uns["processing_steps"]["transformation"] = f"logicle, t={t}, m={m}, w={w}, a={a}"
print(f"all {adata.n_vars} channels in {elapsed / 3600:.2f} hours")

`.raw` is set to the `positives` view before the file is written, because
`scanpy.pl.umap` and `plot_marker` read the intensities from `.raw` and the
logicle values in `.X` have no fixed zero point to threshold against.

In [ ]:
adata.raw = ann.AnnData(
    X=adata.layers["positives"], var=adata.var.copy(), obs=adata.obs.copy()
)
adata.uns["processing_steps"]["raw_values"] = "layers['positives']"

adata.write(OUT_LOGICLE)
print(f"wrote {OUT_LOGICLE}: {adata.n_obs:,} cells")

## Downsample

Every experiment is capped at the same number of cells, so a large experiment
does not dominate the embedding.

In [ ]:
target_size = int(DESIRED_CELL_NUMBER / adata.obs["experiment_number"].nunique())
print(f"cap of {target_size:,} cells per experiment")

subset = downsample(
    adata, obs="experiment_number", target_size=target_size, seed=DOWNSAMPLE_SEED
)
subset.write(OUT_LOGICLE_SUBSET)
print(f"wrote {OUT_LOGICLE_SUBSET}: {subset.n_obs:,} cells")

del adata
gc.collect()

## Step 4, the embedding

`compute_umap` z-scores a copy of the intensities, runs the principal component
analysis on that copy, builds the neighbour graph and runs UMAP.  The returned
object keeps the untransformed intensities in `.X`, so a marker plot still shows
the logicle value.

UMAP is only reproducible for a fixed random state together with fixed versions
of `umap-learn` and `pynndescent`.  Both are pinned in `env/requirements.txt`.

In [ ]:
subset.uns["processing_steps"]["principal_components"] = str(N_PCS)
subset.uns["processing_steps"]["neighbours_umap"] = str(N_NEIGHBORS)

start = time.perf_counter()
subset = compute_umap(
    subset,
    genes_to_exclude=["LARRY"],
    n_pcs=N_PCS,
    n_neighbors=N_NEIGHBORS,
    random_state=EMBEDDING_RANDOM_STATE,
)
print(f"embedding in {(time.perf_counter() - start) / 60:.1f} minutes")

rotate_umap(subset, angle=UMAP_ROTATION)
subset.uns["processing_steps"]["umap_rotation"] = str(UMAP_ROTATION)

subset.write(OUT_UMAP)
print(f"wrote {OUT_UMAP}")

## The embedding, coloured by each marker

In [ ]:
cmap = LinearSegmentedColormap.from_list(
    "marker", ["lightgrey", "salmon", "salmon", "salmon", "orangered", "red"]
)
axes = sc.pl.umap(
    subset, color=MARKERS, ncols=6, wspace=0, hspace=0.2,
    show=False, size=10, vmin=POSITIVE_THRESHOLD, cmap=cmap,
)
for ax in axes:
    ax.set_box_aspect(1)
    ax.title.set_fontsize(20)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

The same markers again as a yes or no call: a cell is red when its scaled
`positives` value is above `POSITIVE_THRESHOLD`.

In [ ]:
plot_marker(subset, markers=MARKERS, threshold=POSITIVE_THRESHOLD)
plt.show()

## Where each experiment sits in the embedding

One panel per experiment, its own cells in colour and every cell in light grey.

In [ ]:
plot_umap_by_group(subset, obs="experiment_number")
plt.show()

In [ ]:
for experiment, wells in subset.obs.groupby("experiment_number", observed=True)["well_id"]:
    print(experiment, sorted(wells.unique()))

## Step 5, Leiden clusters

Resolution 0.9.  A higher resolution gives more, smaller clusters.  The
cluster numbers are not stable across versions of `leidenalg` or `umap-learn`,
so any mapping from a number to a cell type has to be rechecked when those
change.

In [ ]:
sc.tl.leiden(subset, random_state=LEIDEN_SEED, resolution=LEIDEN_RESOLUTION)
subset.uns["processing_steps"]["leiden_resolution"] = str(LEIDEN_RESOLUTION)
print(f"{subset.obs['leiden'].nunique()} clusters")

subset.write(OUT_LEIDEN)
print(f"wrote {OUT_LEIDEN}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sc.pl.umap(subset, ax=ax, color="leiden", size=5, show=False, legend_loc="on data")
ax.set_box_aspect(1)
plt.show()

In [ ]:
plot_leiden_clusters(subset)
plt.show()

## Which markers separate each cluster

A Wilcoxon rank-sum test per cluster against every other cluster.  `pts=True`
also reports the fraction of cells in the cluster that express the marker.

In [ ]:
sc.tl.rank_genes_groups(subset, groupby="leiden", method="wilcoxon", pts=True)
sc.pl.rank_genes_groups(subset, n_genes=20, sharey=False)

In [ ]:
subset.uns["processing_steps"]